<a href="https://colab.research.google.com/github/datascience-uniandes/hypothesis-testing-tutorial/blob/master/hypothesis-testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hypothesis Testing

MINE-4101: Applied Data Science  
Univerisdad de los Andes  
  
**Dataset:** AirBnb Listings - Mexico City, Distrito Federal, Mexico [[dataset](http://insideairbnb.com/get-the-data/) | [dictionary](https://docs.google.com/spreadsheets/d/1iWCNJcSutYqpULSQHlNyGInUvHg2BoUGoNRIGa6Szc4/edit?usp=sharing)]. This dataset comprises information about Airbnb property listings in Mexico City. It includes data points like neighborhood, property type, price per night, number of reviews, review scores, availability, amenities, and more.

**Business Context:** Property Investment and Vacation Rental Strategy. You're a consultant for individuals or firms looking to invest in properties for Airbnb rentals. They want to identify the most lucrative neighborhoods, optimal pricing strategies, and understand the factors that contribute to positive reviews and frequent bookings. <span style="color: red;">Since you currently only have a sample of all the properties listed in the city, you must ensure that the insights you extract from your analysis can be generalized to the entire set of properties.</span>

Last update: September, 2025

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import ttest_ind, shapiro, mannwhitneyu, chi2_contingency

In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

### Load the data

In [ ]:
listings_df = pd.read_csv("./data/listings.csv.gz").sample(frac=0.02, random_state=100)

In [ ]:
listings_df.shape

In [ ]:
listings_df.dtypes

In [ ]:
listings_df.sample(5)

### Transform the data

In [ ]:
listings_df["price"] = listings_df["price"].str.replace("[$,]", "", regex=True).astype(float)

### Remove selected properties

In [ ]:
listings_df["room_type"].value_counts()

In [ ]:
listings_df = listings_df.loc[~listings_df["room_type"].isin(["Hotel room", "Shared room"])]

In [ ]:
listings_df.shape

### Remove some critical outliers based on listing price

In [ ]:
q1 = listings_df["price"].quantile(0.25)
q3 = listings_df["price"].quantile(0.75)
iqr = q3 - q1

In [ ]:
listings_df = listings_df.loc[listings_df["price"] <= (q3 + 3 * iqr)]

In [ ]:
listings_df.shape

### Business question 1

After selecting a couple of neighborhoods with good investment potential, analyze the listing price for those neighborhoods. On average, do any of the neighborhoods have higher prices?

In [ ]:
listings_df["neighbourhood_cleansed"].value_counts(dropna=False)

In [ ]:
selected_neighborhoods = ["Miguel Hidalgo", "Benito Juárez"]

In [ ]:
# Showing some statistics for neighborhoods of interest
listings_df.loc[listings_df["neighbourhood_cleansed"].isin(selected_neighborhoods)].groupby("neighbourhood_cleansed")["price"].describe()

In [ ]:
# Plotting price distribution by neighborhood
fig, ax = plt.subplots(1, 1, figsize=(15, 5))
sns.histplot(
    data=listings_df.loc[listings_df["neighbourhood_cleansed"].isin(selected_neighborhoods)],
    x="price",
    hue="neighbourhood_cleansed",
    bins=40,
    ax=ax
)
for (neighborhood, color) in zip(selected_neighborhoods, ["steelblue", "orange"]):
    ax.axvline(listings_df.loc[listings_df["neighbourhood_cleansed"] == neighborhood, "price"].mean(), color=color, linestyle="dashed", linewidth=2, ymax=0.2)
plt.title("Price distribution by neighborhood (with means)")
plt.show()

<span style="color: red">TODO: Check for normality.</span>

*- - - LET'S ASSUME NORMALITY - - -*

**Step 1.** Define null and alternative hypothesis:

$$
H_0: \mu_1 = \mu_2
$$
$$
H_a: \mu_1 \neq \mu_2
$$  

**Step 2.** Define the significance level desired:

In [ ]:
alpha = 0.01

**Step 3 and 4.** Choose the appropriate test, calculate the statistic, p-value, *CI*, and effect size:

In [ ]:
a = listings_df.loc[listings_df["neighbourhood_cleansed"] == selected_neighborhoods[0], "price"]
b = listings_df.loc[listings_df["neighbourhood_cleansed"] == selected_neighborhoods[1], "price"]

In [ ]:
# Welch’s two-sample t-test
result= ttest_ind(a, b, equal_var=False)

In [ ]:
ci = result.confidence_interval(alpha) # 99% CI for (mean_a - mean_b)

In [ ]:
n1, n2 = len(a), len(b)
m1, m2 = a.mean(), b.mean()
s1, s2 = a.std(), b.std()

sp = np.sqrt(((n1 - 1) * s1 ** 2 + (n2 - 1) * s2 ** 2) / (n1 + n2 - 2)) # Pooled SD
d = (m1 - m2) / sp # Cohen's d

df_pooled = n1 + n2 - 2
J = 1 - 3 / (4 * df_pooled - 1)
g = J * d # Hedges' g (bias-corrected d), particularly important when sample sizes are smal

In [ ]:
print(f"t-statistic: {result.statistic:.4f}")
print(f"p-value:     {result.pvalue:.4g}")
print(f"IC99%:       [{ci.low:.3f}, {ci.high:.3f}]")

print(f"Cohen's d:  {d:.3f}")
print(f"Hedges' g:  {g:.3f}")

**Step 5.** Interpret, make the decision:

In [ ]:
if result.pvalue <= alpha:
    print(f"REJECT THE NULL HYPOTHESIS: The difference in listing price between {selected_neighborhoods[0]} and {selected_neighborhoods[1]} neighbourhoods is statistically significant.")
else:
    print(f"FAIL TO REJECT THE NULL HYPOTHESIS: The difference in listing price between {selected_neighborhoods[0]} and {selected_neighborhoods[1]} neighbourhoods is not statistically significant.")

In [ ]:
reference = [0.2, 0.5, 0.8]
magnitude = min(reference, key=lambda r: abs(r - g))

match magnitude:
    case 0.2:
        print("THE EFFECT SIZE IS SMALL.")
    case 0.5:
        print("THE EFFECT SIZE IS MODERATE.")
    case 0.8:
        print("THE EFFECT SIZE IS LARGE.")

**Potential implication for an investor:**  

Listing prices in Miguel Hidalgo tend to be more expensive (573 monetary units, on average) than in Benito Juárez. Depending on land prices and profit margins, it would be wise to invest in Miguel Hidalgo, where users are willing to pay more money.

### Business question 2

To select the best room type to invest in, are there room types that are more prevalent in certain neighborhoods?

In [ ]:
neighborhood_frec_cumsum = listings_df["neighbourhood_cleansed"].value_counts(dropna=False, normalize=True).cumsum()
neighborhood_frec_cumsum

In [ ]:
# Filtering by Pareto's rule at 90%
most_representative_neighborhoods = neighborhood_frec_cumsum.loc[neighborhood_frec_cumsum < 0.85].index.tolist()
most_representative_neighborhoods

<span style="color: red">TODO: From the begining of the script, use all room types, and collapse rare types into “Other” instead.</span>

In [ ]:
listings_df["room_type"].value_counts(dropna=False, normalize=True)

In [ ]:
contingency_table = pd.crosstab(
    listings_df.loc[listings_df["neighbourhood_cleansed"].isin(most_representative_neighborhoods)]["neighbourhood_cleansed"],
    listings_df.loc[listings_df["neighbourhood_cleansed"].isin(most_representative_neighborhoods)]["room_type"]
)
contingency_table

**Step 1.** Define null and alternative hypothesis:

$$
H_0: \text{The variables are not dependent}
$$
$$
H_a: \text{The variables are dependent}
$$ 

**Step 2.** Define the significance level desired:

In [ ]:
alpha = 0.01

**Step 3 and 4.** Choose the appropriate test, calculate the statistic, p-value, and effect size:

In [ ]:
chi2, pvalue, _, expected = chi2_contingency(contingency_table, correction=False)

In [ ]:
n = contingency_table.values.sum()
r, k = contingency_table.shape

phi2 = chi2 / n
phi2_corr = max(0, phi2 - ((k - 1) * (r - 1)) / (n - 1))
r_corr = r - ((r - 1) ** 2) / (n - 1)
k_corr = k - ((k - 1) ** 2) / (n - 1)

cramers_v = np.sqrt(phi2_corr / max(1e-12, min(r_corr - 1, k_corr - 1))) # Cramér's V, bias corrected

In [ ]:
print(f"chi-square: {chi2:.4f}")
print(f"p-value:    {pvalue:.4g}")

print(f"Cramér's V: {cramers_v:.3f}")

In [ ]:
if pvalue <= alpha:
    print("REJECT THE NULL HYPOTHESIS: There's a statistically significant dependency between neighborhood and room type.")
else:
    print("FAIL TO REJECT THE NULL HYPOTHESIS: There's no statistically significant dependency between neighborhood and room type.")

In [ ]:
reference = [0.1, 0.3, 0.5]
magnitude = min(reference, key=lambda r: abs(r - cramers_v))

match magnitude:
    case 0.1:
        print("THE EFFECT SIZE IS SMALL.")
    case 0.3:
        print("THE EFFECT SIZE IS MODERATE.")
    case 0.5:
        print("THE EFFECT SIZE IS LARGE.")

**Potential implication for an investor:**  

In the neighborhoods with more listings, the ratio between entire home/apts and private rooms is more differentiated. Is gentrification playing a role? Are neighborhoods with fewer listed properties more residentials?